## Load notebook

### Python modules

In [ ]:
# standard libs
import numpy as np
import pandas as pd
from scipy import sparse
import lmfit as lm
from tqdm import tqdm
import io, sys, importlib
from contextlib import redirect_stdout
import warnings
warnings.filterwarnings("ignore", message="Using UFloat objects with std_dev==0*")


# matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
plt.rcParams['font.size'], plt.rcParams['axes.labelsize'] = 14, 18

# XPCS library
sys.path.append('./XPCSlibrary/')

import ID10tools as ID10
from ID10tools import Nx, Ny, Npx
importlib.reload(ID10)
ID10.set_version('v2')

import XPCStools as XPCS
from XPCStools import E2lambda, lambda2E, theta2Q, Q2theta, decorrelation_f
importlib.reload(XPCS)
XPCS.set_beamline('ID10')

import COSMICRAYtools as COSMIC
importlib.reload(COSMIC)
COSMIC.set_beamline('ID10')

#### EXPERIMENTAL VARIABLES ####
XPCS.set_expvar(1350, 1350, 7.1)
ID10.Nfmax_dense_file = 2000
ID10.Nfmax_sparse_file = 5000
################################

###### FOLDER PATHS ######
from folder_paths import *
##########################

### Functions

In [ ]:
def MTplotfit_byQ_4damaged_vGeO2(itime, g2mt, t1_fit, t2_fit, save=True):

    print("Using Global variables: model, params, Qs_section, dQ_section, sample_name, Ndataset, Nscan\n")    
    global model, params, Qs_section, dQ_section, sample_name, Ndataset, Nscan

    if t1_fit is None: t1_fit = 0
    if t2_fit is None: t2_fit = 1e100
    
    g2fit = pd.DataFrame(columns=['Q', 'dQ', 'tau', 'dtau', 'dtau%', 'beta', 'dbeta', 'dbeta%', 'c', 'dc', 'dc%', 'y0', 'dy0', 'dy%', 'redchi2'])

    # DATA PLOTS
    fig, ax = plt.subplots(figsize=(10,5))
    for i, q in enumerate(g2mt.keys()):

        # FIT DATA
        # time-mask 4 fitting
        tmask = (t1_fit<g2mt[q][0])*(g2mt[q][0]<t2_fit)
        x, y, dy =  g2mt[q][0][tmask], g2mt[q][1][tmask], g2mt[q][2][tmask]

        # compute and save the fit
        fit = model.fit(y, params, t=x, method='least_squares', weights=1/dy)
        g2fit.loc[i] = [q, dQ_section,
                        fit.params['tau'].value, fit.params['tau'].stderr, fit.params['tau'].stderr/fit.params['tau'].value*100,
                        fit.params['beta'].value, fit.params['beta'].stderr, fit.params['beta'].stderr/fit.params['beta'].value*100,
                        fit.params['c'].value, fit.params['c'].stderr, fit.params['c'].stderr/fit.params['c'].value*100,
                        fit.params['y0'].value, fit.params['y0'].stderr, fit.params['y0'].stderr/fit.params['y0'].value*100,
                        fit.redchi,]
        
        # PLOT FIT
        ax.errorbar(g2mt[q][0], g2mt[q][1], yerr=g2mt[q][2], fmt = 'o', c='C'+str(i), label =f'Q = {q} $\\AA^{-1}$')
        xfit = np.arange(np.min(g2mt[q][0]), np.max(g2mt[q][0]), 1e-2)
        yfit = fit.eval(t=xfit)
        ax.plot(xfit, yfit, c='C'+str(i), linestyle='--')

    if t1_fit!=0:     plt.axvline(t1_fit, color='red', linestyle='--')
    if t2_fit!=1e100: plt.axvline(t2_fit, color='red', linestyle='--')
    ax.set_xlabel('t [s]')
    ax.set_ylabel('g2')
    ax.set_xscale('log')
    
    if save: g2fit.to_csv(f"{g2fit4damaged_vGeO2_folder}g2fit-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={Qs_section[0]}:{Qs_section[-1]}-dQ={dQ_section}.txt", index=False)
    display(g2fit)

    return g2fit, fig, ax

def plotfitparams_byQ_4damaged_vGeO2(g2fit):
    print("Using Global variables: sample_name, Ndataset, Nscan \n")
    global sample_name, Ndataset, Nscan

    fig, axs = plt.subplots(2, 2, figsize=(14, 8))

    # tau
    axs[0, 0].errorbar(g2fit.Q, g2fit.tau, yerr=g2fit.dtau, fmt='o')
    axs[1, 1].set_xlabel('Q [1/$\\AA$]')
    axs[0, 0].set_ylabel('tau [s]')

    # beta
    axs[0, 1].errorbar(g2fit.Q, g2fit.beta, yerr=g2fit.dbeta, fmt='o')
    axs[1, 1].set_xlabel('Q [1/$\\AA$]')
    axs[0, 1].set_ylabel('beta')

    # c
    axs[1, 0].errorbar(g2fit.Q, g2fit.c, yerr=g2fit.dc, fmt='o')
    axs[1, 1].set_xlabel('Q [1/$\\AA$]')
    axs[1, 0].set_ylabel('c')

    # y0
    axs[1, 1].errorbar(g2fit.Q, g2fit.y0, yerr=g2fit.dy0, fmt='o')
    axs[1, 1].set_xlabel('Q [1/$\\AA$]')
    axs[1, 1].set_ylabel('y0')

    return fig, axs

### Load masks

In [ ]:
### e4m MASKS
e4m_htmask_GeO2_7_30C  = np.load(MASKS_folder+'e4m_htmask-GeO2_7_30C_0003_0003' +'.npy')
e4m_htmask_GeO2_7_100C = np.load(MASKS_folder+'e4m_htmask-GeO2_7_100C_0001_0003'+'.npy')
e4m_htmask_GeO2_7_170C = np.load(MASKS_folder+'e4m_htmask-GeO2_7_170C_0001_0003'+'.npy')
e4m_htmask_GeO2_7_240C = np.load(MASKS_folder+'e4m_htmask-GeO2_7_240C_0001_0003'+'.npy')
e4m_htmask = e4m_htmask_GeO2_7_30C * e4m_htmask_GeO2_7_100C * e4m_htmask_GeO2_7_170C * e4m_htmask_GeO2_7_240C

e4m_mask = np.load(MASKS_folder+'e4m_mask'+'.npy')

plt.figure(figsize=(5, 5))
plt.imshow((e4m_mask*e4m_htmask).reshape(Nx,Ny), cmap='gray', origin='lower')
plt.xlabel('Y [px]')
plt.ylabel('X [px]')
plt.tight_layout(); plt.show()

## XPCS scan: delcoup=1.75, T=30, 75min @ 20ms ON damaged point (GeO2_6, dataset 1, scan 12)
Start of the Temperature ramp!

In [ ]:
#######################################
sample_name = 'GeO2_6'
Ndataset = 1
Nscan =12
Nfi, Nff = 1, None
#######################################

scan = ID10.load_scan(RAW_folder, sample_name, Ndataset, Nscan)
Ei = scan['monoe']
itime = scan['fast_timer_period'][0]
theta = scan['delcoup']
Q = round(XPCS.theta2Q(Ei,  theta),2)

print('#############################')
print('command =', scan['command'])
print('Ei =', Ei)
print('itime =', itime)
print('theta =', theta)
print('Q =', Q)
print('T = {:.2f} min'.format(len(scan['fast_timer_period'])*itime/60))   
print('#############################\n')

e4m_data = ID10.load_sparse_e4m(RAW_folder, sample_name, Ndataset, Nscan, Nfi, Nff,  n_jobs=60)
e4m_data = COSMIC.fast_gamma_filter(e4m_data, Imaxth_high=4)

### Qmask

In [ ]:
###################
Qs_section = [.2, .17, .14]
dQ_section = .015
###################

Qmask = XPCS.gen_Qmask(Ei, theta, Qs_section, dQ_section, Qmap_plot=False)

### Flux check

In [ ]:
geom = [{'geom':'Rectangle', 'x0':1250, 'y0':1300, 'xl':250, 'yl':1950, 'inside':False},
        {'geom':'Circle', 'Cx':1300, 'Cy':950, 'r':1470, 'inside':True}, 
        ]
XPCS.gen_plots4mask(e4m_data, itime, Ith_high=.4, Nff=10000, mask_geom=geom,)
bs_mask = XPCS.gen_mask(mask_geom=geom)

### Intensity analysis

In [ ]:
####################
Lbin = 100
Nstep = 1000
Nfi = 15
####################

It = {}
for q in Qmask.keys():
    mask= e4m_mask * e4m_htmask * bs_mask * Qmask[q]
    It[q] = np.vstack(XPCS.get_It(e4m_data, itime, mask=mask, Nfi=Nfi, Lbin=Lbin, Nstep=Nstep))
    np.savetxt(f"{Intensity_folder}Idt-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={q}-dQ={dQ_section}.txt", It[q])

In [ ]:
plt.figure(figsize=(10,5))
for q in Qmask.keys():
    plt.scatter(It[q][0], It[q][1], label='Q={:.2f}'.format(q), s=1)
plt.xlabel('Time (s)')
plt.ylabel('Integrated intensity [photons/pixel/s]')
plt.gca().secondary_xaxis('top', functions=(lambda x: x/itime, lambda x: x*itime))
plt.legend()
plt.tight_layout(); plt.show()

### Multitau correlation of damaged-vGeO2

In [ ]:
##### INPUTS #####
Nfi = 50_000
Nff = None
sparse_depth = 12
ch_depth = 4
##################

G2tmt, g2mt = {}, {}
for q in tqdm(Qmask.keys(), desc=f'Calculating G2tmt @ q={q}'):
    mask = e4m_mask * e4m_htmask * bs_mask * Qmask[q]
    G2tmt[q] = XPCS.get_G2tmt_4sparse(e4m_data, sparse_depth, ch_depth=ch_depth, mask=mask, Nfi=Nfi, Nff=Nff)
    g2mt[q] = np.vstack(XPCS.get_g2mt(itime, G2tmt[q]))
    np.savetxt(f"{g24damaged_vGeO2_folder}g2mt-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={q}-dQ={dQ_section}.txt", g2mt[q])

In [ ]:
############################
q = .2
vmin, vmax = 1.0, 1.03
lower_corr = 4
############################

XPCS.plot_G2tmt(G2tmt[q], itime, vmin, vmax, yscale='log', lower_corr=lower_corr)

In [ ]:
##################
t1_fit = 0
t2_fit = 1e100
##################

################# KKW MODEL #################
model = lm.Model(decorrelation_f)
params = lm.Parameters()
params.add('tau', value=1, min=0, max=10)
params.add('beta', value=1, min=0, max=2)
params.add('c', value=.03, min=0, max=.1)
params.add('y0', value=1, min=1, max=1.03)
#params.add('y0', value=1.0023, vary=False)
##############################################

g2fit, fig, ax = MTplotfit_byQ_4damaged_vGeO2(itime, g2mt, t1_fit, t2_fit, save=True)
ax.set_ylim(1, 1.035)
ax.legend()
fig.tight_layout(); fig.show()

fig, axs = plotfitparams_byQ_4damaged_vGeO2(g2fit)
fig.tight_layout(); fig.show()

##   XPCS scan: delcoup = 1, T=30, 60min @ 1ms (stopped @ 2600000) (GeO2_6q_delcoup1, dataset 1, scan 2)

**ATTENZIONE!!!**
Le due strisciate in basso sul detector cambiano il tau da 5 a 1 s!!!

In [ ]:
#######################################
sample_name = 'GeO2_6q_delcoup_1'
Ndataset = 1
Nscan = 2
Nfi, Nff = None, 2600000 # NO BEAM @ 2600000 frames
#######################################

scan = ID10.load_scan(RAW_folder, sample_name, Ndataset, Nscan)
Ei = scan['monoe']
itime = scan['fast_timer_period'][0]
theta = scan['delcoup']
Q = round(XPCS.theta2Q(Ei,  theta),2)

print('#############################')
print('command =', scan['command'])
print('Ei =', Ei)
print('itime =', itime)
print('theta =', theta)
print('Q =', Q)
print('T = {:.2f} min'.format(len(scan['fast_timer_period'])*itime/60))   
print('#############################\n')

e4m_data = ID10.load_sparse_e4m(RAW_folder, sample_name, Ndataset, Nscan, Nfi, Nff,  n_jobs=50)
e4m_data = COSMIC.fast_gamma_filter(e4m_data, Imaxth_high=4)

### Qmask

In [ ]:
###################
Qs_section = [.085, .105, .125, .145,]
dQ_section = .01
###################

Qmask = XPCS.gen_Qmask(Ei, theta, Qs_section, dQ_section, Qmap_plot=False)

### Beamstop mask

In [ ]:
geom = [{'geom':'Rectangle', 'x0':1250, 'y0':1300, 'xl':250, 'yl':1950, 'inside':False},
        {'geom':'Rectangle', 'x0':150, 'y0':1400, 'xl':600, 'yl':2000, 'inside':False},
        {'geom':'Circle', 'Cx':1300, 'Cy':950, 'r':1470, 'inside':True}, 
        ]
XPCS.gen_plots4mask(e4m_data, itime, Ith_low = .14, Ith_high=.23, Nff=300000, mask_geom=geom,)
bs_mask = XPCS.gen_mask(mask_geom=geom)

### Intensity analysis

In [ ]:
####################
Lbin = 100
Nstep = 1000
Nfi = 15
Nff=None
####################

It = {}
for q in Qmask.keys():
    mask= e4m_mask * e4m_htmask * bs_mask * Qmask[q]
    It[q] = np.vstack(XPCS.get_It(e4m_data, itime, mask=mask, Nfi=Nfi, Nff=Nff, Lbin=Lbin, Nstep=Nstep))
    np.savetxt(f"{Intensity_folder}Idt-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={q}-dQ={dQ_section}.txt", It[q])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for q in Qmask.keys():
    ax.scatter(It[q][0], It[q][1], label='Q={:.2f}'.format(q), s=1)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Integrated intensity [photons/pixel/s]')
ax.secondary_xaxis('top', functions=(lambda x: x/itime, lambda x: x*itime))
ax.grid()
#ax.set_xlim(0, 100)
ax.legend()
fig.tight_layout()
plt.show()

### Multitau correlation of damaged-vGeO2

In [ ]:
##### INPUTS #####
Nfi = int(.5e6)
Nff = None
sparse_depth = 14
ch_depth = 4
##################

G2tmt, g2mt = {}, {}
for q in tqdm(Qmask.keys(), desc=f'Calculating G2tmt @ q={q}'):
    mask = e4m_mask * e4m_htmask * bs_mask * Qmask[q]
    G2tmt[q] = XPCS.get_G2tmt_4sparse(e4m_data, sparse_depth, ch_depth=ch_depth, mask=mask, Nfi=Nfi, Nff=Nff)
    g2mt[q] = np.vstack(XPCS.get_g2mt(itime, G2tmt[q]))
    np.savetxt(f"{g24damaged_vGeO2_folder}g2mt-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={q}-dQ={dQ_section}.txt", g2mt[q])

In [ ]:
############################
q = .105
vmin, vmax = 1.0, 1.03
lower_corr = 6
filter_layer = 6
############################

XPCS.plot_G2tmt(G2tmt[q], itime, vmin, vmax, yscale='log', lower_corr=lower_corr, filter_layer=filter_layer)

In [ ]:
##################
t1_fit = 0
t2_fit = 1e100
##################

################# KKW MODEL #################
model = lm.Model(decorrelation_f)
params = lm.Parameters()
params.add('tau', value=1, min=0, max=10)
params.add('beta', value=1, min=0, max=2)
params.add('c', value=.03, min=0, max=.1)
params.add('y0', value=1, min=1, max=1.03)
#params.add('y0', value=1.0023, vary=False)
##############################################

g2fit, fig, ax = MTplotfit_byQ_4damaged_vGeO2(itime, g2mt, t1_fit, t2_fit, save=True)
ax.set_ylim(1, 1.045)
ax.legend()
fig.tight_layout(); fig.show()

fig, axs = plotfitparams_byQ_4damaged_vGeO2(g2fit)
fig.tight_layout(); fig.show()

## XPCS scan: delcoup=3, T=30 60min @ 1ms (GeO2_6q_delcoup3, dataset 1, scan 2)

In [ ]:
#######################################
sample_name = 'GeO2_6q_delcoup_3'
Ndataset = 1
Nscan = 2
Nfi, Nff = None, None
#######################################

scan = ID10.load_scan(RAW_folder, sample_name, Ndataset, Nscan)
Ei = scan['monoe']
itime = scan['fast_timer_period'][0]
theta = scan['delcoup']
Q = round(XPCS.theta2Q(Ei,  theta),2)

print('#############################')
print('command =', scan['command'])
print('Ei =', Ei)
print('itime =', itime)
print('theta =', theta)
print('Q =', Q)
print('T = {:.2f} min'.format(len(scan['fast_timer_period'])*itime/60))   
print('#############################\n')

e4m_data = ID10.load_sparse_e4m(RAW_folder, sample_name, Ndataset, Nscan, Nfi, Nff,  n_jobs=1)
e4m_data = COSMIC.fast_gamma_filter(e4m_data, Imaxth_high=4)

### Qmask

In [ ]:
###################
Qs_section = [.3,.26]
dQ_section = .02
###################

Qmask = XPCS.gen_Qmask(Ei, theta, Qs_section, dQ_section, Qmap_plot=False)

### Beamstop mask

In [ ]:
geom = [{'geom':'Rectangle', 'x0':1250, 'y0':1300, 'xl':250, 'yl':1950, 'inside':False},
        {'geom':'Rectangle', 'x0':250, 'y0':1550, 'xl':470, 'yl':2000, 'inside':False},
        {'geom':'Circle', 'Cx':1300, 'Cy':950, 'r':1470, 'inside':True}, 
        ]
XPCS.gen_plots4mask(e4m_data, itime, Ith_low = .12, Ith_high=.25, Nff=300000, mask_geom=geom,)
bs_mask = XPCS.gen_mask(mask_geom=geom)

### Intensity analysis

In [ ]:
####################
Lbin = 100
Nstep = 1000
Nfi = 15
Nff=None
####################

It = {}
for q in Qmask.keys():
    mask= e4m_mask * e4m_htmask * bs_mask * Qmask[q]
    It[q] = np.vstack(XPCS.get_It(e4m_data, itime, mask=mask, Nfi=Nfi, Nff=Nff, Lbin=Lbin, Nstep=Nstep))
    np.savetxt(f"{Intensity_folder}Idt-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={q}-dQ={dQ_section}.txt", It[q])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for q in Qmask.keys():
    ax.scatter(It[q][0], It[q][1], label='Q={:.2f}'.format(q), s=1)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Integrated intensity [photons/pixel/s]')
ax.secondary_xaxis('top', functions=(lambda x: x/itime, lambda x: x*itime))
ax.grid()
#ax.set_xlim(0, 100)
ax.legend()
fig.tight_layout()
plt.show()

### Multitau correlation of damaged-vGeO2

In [ ]:
##### INPUTS #####
Nfi = int(.5e6)
Nff = None
sparse_depth = 14
ch_depth = 4
##################

G2tmt, g2mt = {}, {}
for q in tqdm(Qmask.keys(), desc=f'Calculating G2tmt @ q={q}'):
    mask = e4m_mask * e4m_htmask * bs_mask * Qmask[q]
    G2tmt[q] = XPCS.get_G2tmt_4sparse(e4m_data, sparse_depth, ch_depth=ch_depth, mask=mask, Nfi=Nfi, Nff=Nff)
    g2mt[q] = np.vstack(XPCS.get_g2mt(itime, G2tmt[q]))
    np.savetxt(f"{g24damaged_vGeO2_folder}g2mt-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={q}-dQ={dQ_section}.txt", g2mt[q])

In [ ]:
############################
q = .3
vmin, vmax = 1.0, 1.03
lower_corr = 6
filter_layer = 6
############################

XPCS.plot_G2tmt(G2tmt[q], itime, vmin, vmax, yscale='log', lower_corr=lower_corr, filter_layer=filter_layer)

In [ ]:
##################
t1_fit = 0
t2_fit = 1e100
##################

################# KKW MODEL #################
model = lm.Model(decorrelation_f)
params = lm.Parameters()
params.add('tau', value=1, min=0, max=10)
params.add('beta', value=1, min=0, max=2)
params.add('c', value=.03, min=0, max=.1)
params.add('y0', value=1, min=1, max=1.03)
#params.add('y0', value=1.0023, vary=False)
##############################################

g2fit, fig, ax = MTplotfit_byQ_4damaged_vGeO2(itime, g2mt, t1_fit, t2_fit, save=True)
ax.set_ylim(1, 1.035)
ax.legend()
fig.tight_layout(); fig.show()

fig, axs = plotfitparams_byQ_4damaged_vGeO2(g2fit)
fig.tight_layout(); fig.show()

##   XPCS scan delcoup=5, T=30, 60min @ 1ms (GeO2_6q_delcoup5, dataset 1, scan 2)

In [ ]:
#######################################
sample_name = 'GeO2_6q_delcoup_5'
Ndataset = 1
Nscan = 2
Nfi, Nff = None, None
#######################################

scan = ID10.load_scan(RAW_folder, sample_name, Ndataset, Nscan)
Ei = scan['monoe']
itime = scan['fast_timer_period'][0]
theta = scan['delcoup']
Q = round(XPCS.theta2Q(Ei,  theta),2)

print('#############################')
print('command =', scan['command'])
print('Ei =', Ei)
print('itime =', itime)
print('theta =', theta)
print('Q =', Q)
print('T = {:.2f} min'.format(len(scan['fast_timer_period'])*itime/60))   
print('#############################\n')

e4m_data = ID10.load_sparse_e4m(RAW_folder, sample_name, Ndataset, Nscan, Nfi, Nff,  n_jobs=60)
e4m_data = COSMIC.fast_gamma_filter(e4m_data, Imaxth_high=4)

### Qmask

In [ ]:
###################
Qs_section = [.415,.455]
dQ_section = .02
###################

Qmask = XPCS.gen_Qmask(Ei, theta, Qs_section, dQ_section, Qmap_plot=False)

### Beamstop mask

In [ ]:
geom = [{'geom':'Rectangle', 'x0':1250, 'y0':1300, 'xl':250, 'yl':1950, 'inside':False},
        {'geom':'Rectangle', 'x0':0, 'y0':400, 'xl':3000, 'yl':3000, 'inside':True},
        {'geom':'Circle', 'Cx':1300, 'Cy':950, 'r':1470, 'inside':True}, 
        ]
XPCS.gen_plots4mask(e4m_data, itime, Ith_low = .15, Ith_high=.9, Nff=10000, mask_geom=geom,)
bs_mask = XPCS.gen_mask(mask_geom=geom)

### Intensity analysis

In [ ]:
####################
Lbin = 100
Nstep = 1000
Nfi = 15
Nff=None
####################

It = {}
for q in Qmask.keys():
    mask= e4m_mask * e4m_htmask * bs_mask * Qmask[q]
    It[q] = np.vstack(XPCS.get_It(e4m_data, itime, mask=mask, Nfi=Nfi, Nff=Nff, Lbin=Lbin, Nstep=Nstep))
    np.savetxt(f"{Intensity_folder}Idt-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={q}-dQ={dQ_section}.txt", It[q])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for q in Qmask.keys():
    ax.scatter(It[q][0], It[q][1], label='Q={:.2f}'.format(q), s=1)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Integrated intensity [photons/pixel/s]')
ax.secondary_xaxis('top', functions=(lambda x: x/itime, lambda x: x*itime))
ax.grid()
#ax.set_xlim(0, 100)
ax.legend()
fig.tight_layout()
plt.show()

### Multitau correlation of damaged-vGeO2

In [ ]:
##### INPUTS #####
Nfi = int(.5e6)
Nff = None
sparse_depth = 14
ch_depth = 4
##################

G2tmt, g2mt = {}, {}
for q in tqdm(Qmask.keys(), desc=f'Calculating G2tmt @ q={q}'):
    mask = e4m_mask * e4m_htmask * bs_mask * Qmask[q]
    G2tmt[q] = XPCS.get_G2tmt_4sparse(e4m_data, sparse_depth, ch_depth=ch_depth, mask=mask, Nfi=Nfi, Nff=Nff)
    g2mt[q] = np.vstack(XPCS.get_g2mt(itime, G2tmt[q]))
    np.savetxt(f"{g24damaged_vGeO2_folder}g2mt-{sample_name}_{str(Ndataset).zfill(4)}_{str(Nscan).zfill(4)}-Q={q}-dQ={dQ_section}.txt", g2mt[q])

In [ ]:
############################
q = 0.415
vmin, vmax = 1.0, 1.015
lower_corr = 6
filter_layer = 6
############################

XPCS.plot_G2tmt(G2tmt[q], itime, vmin, vmax, yscale='log', lower_corr=lower_corr, filter_layer=filter_layer)

In [ ]:
##################
t1_fit = 0
t2_fit = 1e100
##################

################# KKW MODEL #################
model = lm.Model(decorrelation_f)
params = lm.Parameters()
params.add('tau', value=1, min=0, max=10)
params.add('beta', value=1, min=0, max=2)
params.add('c', value=.03, min=0, max=.1)
params.add('y0', value=1, min=1, max=1.03)
#params.add('y0', value=1.0023, vary=False)
##############################################

g2fit, fig, ax = MTplotfit_byQ_4damaged_vGeO2(itime, g2mt, t1_fit, t2_fit, save=True)
ax.set_ylim(1, 1.015)
ax.legend()
fig.tight_layout(); fig.show()

fig, axs = plotfitparams_byQ_4damaged_vGeO2(g2fit)
fig.tight_layout(); fig.show()